# Notebook 04 — Final Evaluation & Report

**Prerequisite:** Notebooks 00, 02, and 03 must have finished.

**Inputs:** All `results/*.json` files produced by earlier notebooks  
**Outputs:**
- `results/final_comparison.json`
- `results/comparison_table.csv`
- `results/qualitative_examples.md`
- `results/report_skeleton.md`

**Runtime:** < 5 minutes. No GPU needed.

## Cell 1 — Paths setup (no install needed)

In [ ]:
import os, sys, json, csv
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
if KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working/daa-helper")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

HF_USERNAME = "mimadraza"

from utils.io_helpers import load_json
RESULTS_DIR = PROJECT_ROOT / "results"
print(f"Results dir: {RESULTS_DIR}")
print(f"Files found: {sorted(p.name for p in RESULTS_DIR.glob('*.json'))}")

## Cell 2 — Load all results

In [ ]:
def load_stage(stage):
    pattern = "baseline_*.json" if stage == "baseline" else f"{stage}_*.json"
    files   = sorted(RESULTS_DIR.glob(pattern))
    files   = [f for f in files if "winner" not in f.stem]
    return [json.loads(f.read_text()) for f in files]

def load_winner(stage):
    p = RESULTS_DIR / f"{stage}_winner.json"
    return json.loads(p.read_text()) if p.exists() else {}

baseline_results = load_stage("baseline")
sft_results      = load_stage("sft")
dpo_results      = load_stage("dpo")
sft_winner       = load_winner("sft")
dpo_winner       = load_winner("dpo")

print(f"Baseline: {len(baseline_results)} | SFT: {len(sft_results)} | DPO: {len(dpo_results)}")
print(f"SFT winner: {sft_winner.get('winning_trial', '(not found)')}")
print(f"DPO winner: {dpo_winner.get('winning_trial', '(not found)')}")

## Cell 3 — Build comparison table

In [ ]:
rows = []

for r in baseline_results:
    agg = r["evaluation"]["aggregate"]
    rows.append({
        "stage": "baseline", "trial": r.get("trial_name", "base_model"),
        "mean_bleu": round(agg["mean_bleu"], 2),
        "bertscore_f1": round(agg["mean_bertscore_f1"], 4),
        "combined": round(agg["combined_score"], 4),
        "train_loss": "", "eval_loss": "",
        "description": r.get("config", {}).get("model", ""),
    })

for r in sft_results:
    agg = r["evaluation"]["aggregate"]
    tm  = r.get("training_metrics", {})
    rows.append({
        "stage": "sft", "trial": r["trial_name"],
        "mean_bleu": round(agg["mean_bleu"], 2),
        "bertscore_f1": round(agg["mean_bertscore_f1"], 4),
        "combined": round(agg["combined_score"], 4),
        "train_loss": round(tm.get("train_loss", 0), 4),
        "eval_loss":  round(tm.get("eval_loss",  0), 4),
        "description": r.get("config", {}).get("description", ""),
    })

for r in dpo_results:
    agg = r["evaluation"]["aggregate"]
    tm  = r.get("training_metrics", {})
    rows.append({
        "stage": "dpo", "trial": r["trial_name"],
        "mean_bleu": round(agg["mean_bleu"], 2),
        "bertscore_f1": round(agg["mean_bertscore_f1"], 4),
        "combined": round(agg["combined_score"], 4),
        "train_loss": round(tm.get("train_loss", 0), 4),
        "eval_loss":  round(tm.get("eval_loss",  0), 4),
        "description": r.get("config", {}).get("description", ""),
    })

print(f"{'Stage':<10} {'Trial':<30} {'BLEU':>6} {'BERTScore':>10} {'Combined':>10}")
for row in rows:
    print(f"{row['stage']:<10} {row['trial']:<30} {row['mean_bleu']:>6} "
          f"{row['bertscore_f1']:>10} {row['combined']:>10}")

## Cell 4 — Save comparison_table.csv & final_comparison.json

In [ ]:
# CSV
csv_path = RESULTS_DIR / "comparison_table.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved → {csv_path}")

# JSON
comparison = {"all_rows": rows, "sft_winner": sft_winner, "dpo_winner": dpo_winner}
fc_path = RESULTS_DIR / "final_comparison.json"
fc_path.write_text(json.dumps(comparison, indent=2))
print(f"Saved → {fc_path}")

## Cell 5 — Build qualitative_examples.md

In [ ]:
def get_samples(filename):
    try:
        r = load_json(filename, base_dir="results")
        return {s["prompt"]: s for s in r.get("sample_responses", [])}
    except FileNotFoundError:
        return {}

baseline_samples = {}
for r in baseline_results:
    for s in r.get("sample_responses", []):
        baseline_samples[s["prompt"]] = s

sft_name    = sft_winner.get("winning_trial", "")
dpo_name    = dpo_winner.get("winning_trial", "")
sft_samples = get_samples(f"sft_{sft_name}.json") if sft_name else {}
dpo_samples = get_samples(f"dpo_{dpo_name}.json") if dpo_name else {}

common = [p for p in baseline_samples if p in sft_samples and p in dpo_samples][:3]
if not common:
    common = list(baseline_samples.keys())[:3]

lines = ["# Qualitative Examples\n\n",
         "Three prompts shown side-by-side: base model → SFT winner → DPO winner.\n"]
for i, prompt in enumerate(common, 1):
    lines.append(f"\n---\n\n## Example {i}\n\n**Prompt:**\n> {prompt}\n")
    lines.append(f"\n### Base Model\n{baseline_samples.get(prompt,{}).get('response','_(not found)_')}\n")
    lines.append(f"\n### SFT Winner ({sft_name})\n{sft_samples.get(prompt,{}).get('response','_(not found)_')}\n")
    lines.append(f"\n### DPO Winner ({dpo_name})\n{dpo_samples.get(prompt,{}).get('response','_(not found)_')}\n")

qual_path = RESULTS_DIR / "qualitative_examples.md"
qual_path.write_text("".join(lines), encoding="utf-8")
print(f"Saved → {qual_path}")

## Cell 6 — Build report_skeleton.md

In [ ]:
sft_m = sft_winner.get("winning_metrics", {})
dpo_m = dpo_winner.get("winning_metrics", {})
b_row = next((r for r in rows if r["stage"] == "baseline"), {})

b_combined = b_row.get("combined", 0)
s_combined = sft_m.get("combined_score", 0)
d_combined = dpo_m.get("combined_score", 0)

table_rows = "\n".join(
    f"| {r['stage']} | {r['trial']} | {r['mean_bleu']} | {r['bertscore_f1']} "
    f"| {r['combined']} | {r['eval_loss'] or '—'} |"
    for r in rows
)

skeleton = f"""# DAA Helper — NLP Assignment 04 Report

## Team
[FILL: Member 1 Name (Roll)] — [FILL: Member 2 Name (Roll)]

---

## 1  Introduction
[FILL: 2-3 sentences — Socratic DAA tutoring goal, why TinyLlama, why SFT→DPO.]

---

## 2  Dataset & Gold Answers
- **SFT training data:** `open-r1/codeforces-cots` (solutions_w_editorials), 3 000 samples.
- **DPO preference data:** `trl-lib/ultrafeedback_binarized`, 2 000 pairs.
- **Evaluation set:** 10 DAA prompts; gold answers generated via Claude.ai with the Socratic tutor instruction.

[FILL: data quality observations.]

---

## 3  Methodology

### 3.1  Base Model
TinyLlama/TinyLlama_v1.1, loaded in 4-bit NF4 quantization (QLoRA).

### 3.2  SFT
Five LoRA configurations (Table 1). Winner: highest combined BLEU+BERTScore, tie-broken by lowest eval loss.

### 3.3  DPO
Five DPO trials varying β, learning rate, and epochs on top of the SFT winner.

---

## 4  Results

### Table 1 — All Trials

| Stage | Trial | BLEU | BERTScore F1 | Combined | Eval Loss |
|-------|-------|------|--------------|----------|-----------|
{table_rows}

**SFT winner:** {sft_winner.get('winning_trial', '[SFT WINNER]')}  
**DPO winner:** {dpo_winner.get('winning_trial', '[DPO WINNER]')}

### Key Numbers
- Baseline combined score : {b_combined:.4f}
- After SFT               : {s_combined:.4f}  (Δ {s_combined - b_combined:+.4f})
- After DPO               : {d_combined:.4f}  (Δ {d_combined - b_combined:+.4f} vs baseline, {d_combined - s_combined:+.4f} vs SFT)

---

## 5  Qualitative Analysis
[FILL: 2-3 paragraphs from qualitative_examples.md. Did SFT improve Socratic reasoning?
Did DPO further align tone? Any failure modes?]

---

## 6  Discussion & Failure Modes
[FILL: MLP-only trial underperformance, high-LR DPO divergence, any OOMs,
whether BLEU+BERTScore adequately captures Socratic quality.]

---

## 7  Conclusion
[FILL: 2-3 sentences — what worked, what didn't, one future direction.]

---

## Artifacts
- GitHub: [FILL link]
- HF Hub (SFT winner): https://huggingface.co/{HF_USERNAME}/daa-helper-tinyllama-sft-winner
- HF Hub (DPO winner): https://huggingface.co/{HF_USERNAME}/daa-helper-tinyllama-dpo-winner
"""

skel_path = RESULTS_DIR / "report_skeleton.md"
skel_path.write_text(skeleton, encoding="utf-8")
print(f"Saved → {skel_path}")
print("\nOpen results/report_skeleton.md, fill in the [FILL] sections.")
print("Convert to Word with:  pandoc results/report_skeleton.md -o YourName1_YourName2.docx")

## Cell 7 — (Optional) Push results to GitHub

In [ ]:
import subprocess

if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    GITHUB_TOKEN = sec.get_secret("GITHUB_TOKEN")
    GITHUB_USER  = sec.get_secret("GITHUB_USER")

    cmds = [
        ["git", "config", "--global", "user.email", "you@example.com"],
        ["git", "config", "--global", "user.name", GITHUB_USER],
        ["git", "-C", str(PROJECT_ROOT), "add", "results/"],
        ["git", "-C", str(PROJECT_ROOT), "commit", "-m", "final evaluation"],
        ["git", "-C", str(PROJECT_ROOT), "push",
         f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True)
        out = (r.stdout or r.stderr).strip()
        if out: print(out)
else:
    print("Not on Kaggle — skipping push.")

print("\n✓ Notebook 04 complete. Fill in report_skeleton.md and submit.")